# End-to-End Training & Evaluation: Fruit and Vegetable Classifier

This notebook provides a complete walk-through of the two-phase transfer learning workflow, evaluation metrics, and Grad-CAM interpretability for the 36-class produce recognition model.

### Workflow Outline
1. Hardware verification (CUDA / MPS / CPU)
2. PyTorch DataLoaders with specialized augmentations
3. EfficientNet-B0 architecture construction and custom head attachment
4. Two-phase training: Phase 1 (Feature Extraction) & Phase 2 (Fine-Tuning)
5. Comprehensive test evaluation: Confusion matrix and classification report
6. Grad-CAM visual explainability generation
7. Single-image inference demonstration


In [ ]:
# Setup and Device Configuration
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

# Add repository root to path for local src modules
repo_root = Path('..').resolve() if Path('../src').exists() else Path('.').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Computation Device: {device}')
if device.type == 'cuda':
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')


## 1. Load DataLoaders
Initialize PyTorch DataLoaders for train, validation, and test splits.


In [ ]:
from src.dataset import build_dataloaders

train_dir = repo_root / 'data' / 'train'
val_dir   = repo_root / 'data' / 'validation'
test_dir  = repo_root / 'data' / 'test'

train_loader, val_loader, test_loader, class_to_idx = build_dataloaders(
    train_dir=train_dir,
    val_dir=val_dir,
    test_dir=test_dir,
    image_size=224,
    batch_size=32,
    num_workers=2,
)

idx_to_class = {v: k for k, v in class_to_idx.items()}
print(f'Total Classes Loaded    : {len(class_to_idx)}')
print(f'Training Batches        : {len(train_loader)}')
print(f'Validation Batches      : {len(val_loader)}')
print(f'Test Batches            : {len(test_loader)}')


## 2. Model Architecture: EfficientNet-B0 Transfer Learning
Construct the `ProduceClassifier` model with pre-trained ImageNet backbone and task-specific classification head.


In [ ]:
from src.model import build_model

model, _ = build_model(num_classes=36, dropout=0.3, device=device)
summary = model.param_summary()

print(f'Total Parameters    : {summary["total"]:,}')
print(f'Trainable (Phase 1) : {summary["trainable"]:,}')
print(f'Frozen (Phase 1)    : {summary["frozen"]:,}')


## 3. Two-Phase Transfer Learning Regimen
- **Phase 1 (Feature Extraction):** Backbone frozen, train classification head with `lr = 1e-3` (AdamW).
- **Phase 2 (Fine-Tuning):** Unfreeze stages 7 & 8 with `lr = 1e-4` and Cosine Annealing learning rate schedule.


In [ ]:
# Load trained checkpoint if available, or demonstrate training loop execution
checkpoint_path = repo_root / 'results' / 'best_model.pth'

if checkpoint_path.exists():
    print(f'Loading existing trained checkpoint from: {checkpoint_path}')
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    print('Checkpoint loaded successfully!')
else:
    print('Run `python -m src.train` in terminal to execute full two-phase training.')


## 4. Model Evaluation on Test Set
Evaluate the trained model on 359 unseen test samples across all 36 classes.


In [ ]:
from src.evaluate import get_predictions
from sklearn.metrics import classification_report, accuracy_score

y_true, y_pred, confidences = get_predictions(model, test_loader, device)
acc = accuracy_score(y_true, y_pred)
print(f'Overall Test Accuracy: {acc * 100:.2f}%\n')

class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


## 5. Model Explainability with Grad-CAM
Inspect which visual features in the produce image most influenced the model's prediction.


In [ ]:
from src.gradcam import generate_gradcam
from PIL import Image

# Select test image
test_sample = next((test_dir / 'banana').glob('*.*'))
input_img = Image.open(test_sample).convert('RGB')

orig_crop, overlay, filter_only, cam_mat = generate_gradcam(
    image=input_img,
    model=model,
    target_class_idx=class_to_idx['banana'],
    device=device,
    colormap_name='JET',
    alpha=0.5,
)

import matplotlib.pyplot as plt
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5), dpi=150)
ax1.imshow(orig_crop)
ax1.set_title('1. Preprocessed Input Crop', fontweight='bold')
ax1.axis('off')

ax2.imshow(filter_only)
ax2.set_title('2. Filter Only (Activation Heatmap)', fontweight='bold')
ax2.axis('off')

ax3.imshow(overlay)
ax3.set_title('3. Grad-CAM Blended Overlay', fontweight='bold')
ax3.axis('off')

plt.tight_layout()
plt.show()


## 6. Single-Image Inference Demonstration


In [ ]:
from src.predict import predict

top_preds = predict(input_img, model, idx_to_class, device, top_k=5)
print('Predictions for sample image:')
for rank, (cls_name, prob) in enumerate(top_preds, 1):
    print(f'  {rank}. {cls_name:<16} : {prob*100:.2f}%')
